[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanathi/Astro_Gateway_2026/blob/main/Notebooks/02_Spectra.ipynb)

# 02 · สเปกตรัม: อุณหภูมิ ธาตุ และความเร็วของวัตถุท้องฟ้า
**Astro Gateway** · คาบ 11.00–12.00 น.

แสงจากดาวที่แยกตามความยาวคลื่น (สเปกตรัม) บอกเราได้ว่า
- **ร้อนแค่ไหน** — จากชนิดสเปกตรัมและสี เทียบกับดาวมาตรฐานที่รู้อุณหภูมิแล้ว
- **ประกอบด้วยอะไร** — จากเส้นสเปกตรัมที่เป็นลายนิ้วมือของธาตุ
- **เคลื่อนที่เร็วแค่ไหน** — จากการเลื่อนของเส้นสเปกตรัม (ปรากฏการณ์ดอปเพลอร์)

> 📡 **ข้อมูลที่ใช้ (ข้อมูลจริงทั้งหมด):** ส่วน A สเปกตรัมดาว 3 ดวงจาก X-shooter Spectral Library (XSL DR2, Gonneau et al. 2020)
> ส่วน B สเปกตรัมกาแล็กซี SDSS J140102.07+025242.5 จากโครงการ Sloan Digital Sky Survey

> ✏️ **ช่อง `____` ในโค้ดคือส่วนที่ต้องเติมเอง** (สูตรอยู่ในข้อความด้านบนหรือในสไลด์) — ถ้ารันแล้วเจอ `NameError: name '____' is not defined` แปลว่ายังเติมไม่ครบ
>
> 📁 ข้อมูลอยู่ที่ [github.com/thanathi/Astro_Gateway_2026](https://github.com/thanathi/Astro_Gateway_2026) โฟลเดอร์ Data —
> โน้ตบุ๊กจะดาวน์โหลดไฟล์ที่ต้องใช้ให้อัตโนมัติ ไม่ต้องตั้งค่าอะไร (ใช้ไฟล์ของตัวเองได้ด้วยการเปลี่ยนชื่อไฟล์ในโค้ด)
> 💾 เก็บงานของตัวเอง: **File → Save a copy in Drive** ก่อนเริ่มทำ

In [ ]:
#@title 🔧 ฟังก์ชันช่วย — กดรันเซลล์นี้ก่อน (ไม่ต้องแก้ไข) { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
COURSE = "Astro_Gateway_2026"     # ชื่อโฟลเดอร์รายวิชา (GitHub: thanathi/Astro_Gateway_2026)
DATA_URL = "https://raw.githubusercontent.com/thanathi/Astro_Gateway_2026/main/Data/"   # ข้อมูลทั้งหมดบน GitHub · None = ใช้โฟลเดอร์ใน Google Drive แทน
DATA_DIRS = [".", "Data", "../Data", "/content/Data"]


def find_course(root="/content/drive"):
    """หาโฟลเดอร์ COURSE ใน Google Drive: ชั้นบนสุดของ MyDrive, ลึกลงไป 1-2 ชั้น หรือใน Shared drives
    เลือกโฟลเดอร์ที่มี Data อยู่ข้างใน (กันกรณีแตกไฟล์ zip แล้วได้โฟลเดอร์ซ้อนกัน 2 ชั้น)"""
    import glob, os
    hits = []
    for pat in ("MyDrive/{c}", "MyDrive/*/{c}", "MyDrive/*/*/{c}", "Shareddrives/*/{c}"):
        hits += sorted(glob.glob(f"{root}/{pat.format(c=COURSE)}"))
    good = [h for h in hits if os.path.isdir(os.path.join(h, "Data"))]
    return (good or hits or [None])[0]


def connect_drive():
    """ใน Colab: ถ้าใช้ข้อมูลจาก GitHub (DATA_URL) ไม่ต้องทำอะไร — ไฟล์จะดาวน์โหลดเองเมื่อใช้
    ถ้า DATA_URL = None: เชื่อม Google Drive แล้วย้ายไปที่โฟลเดอร์ Notebooks ของรายวิชา ให้ ../Data ใช้ได้
    นอก Colab ไม่ทำอะไร"""
    import os
    try:
        from google.colab import drive
    except ImportError:
        return
    if DATA_URL:
        print("ข้อมูลจะดาวน์โหลดจาก GitHub ให้อัตโนมัติเมื่อใช้ (ไม่ต้องเชื่อม Google Drive)")
        return
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    course = find_course()
    if course is None:
        print(f"⚠️ ไม่พบโฟลเดอร์ {COURSE} ใน Google Drive — โน้ตบุ๊กจะดาวน์โหลดข้อมูลจากคลังข้อมูลต้นทางแทน")
        print(f"   (คัดลอกโฟลเดอร์ {COURSE} ไว้ใน My Drive ตามวิธีใน README.txt แล้วรันเซลล์นี้ใหม่)")
        return
    nb = os.path.join(course, "Notebooks")
    os.chdir(nb if os.path.isdir(nb) else course)
    DATA_DIRS.insert(0, os.path.join(course, "Data"))
    print(f"เชื่อม Google Drive แล้ว: ใช้ข้อมูลจาก {os.path.join(course, 'Data')}")


def data_path(rel, url=None):
    """หาไฟล์ข้อมูล rel (เช่น "SDSS/spec-0531-52028-0573.fits"): ดูในโฟลเดอร์ Data ก่อน
    (หรือไฟล์ที่อัปโหลดในแถบ Files ของ Colab) ถ้าไม่มี ให้ดาวน์โหลดจาก url (คลังข้อมูลต้นทาง)"""
    import os
    name = os.path.basename(rel)
    for base in DATA_DIRS:
        for p in (os.path.join(base, rel), os.path.join(base, name)):
            if os.path.exists(p):
                return p
    for p in (os.path.join("/content", name),):                 # ไฟล์ที่ดาวน์โหลด/อัปโหลดไว้แล้วใน Colab
        if os.path.exists(p):
            return p
    # เขียนไฟล์ที่ดาวน์โหลดไว้ที่ /content ใน Colab (โฟลเดอร์ที่แชร์แบบดูอย่างเดียวเขียนไม่ได้)
    out = os.path.join("/content" if os.path.isdir("/content") else ".", name)
    if DATA_URL and not os.path.isdir("/content"):          # นอก Colab: ไม่ดาวน์โหลดทับโฟลเดอร์ทำงาน
        out = os.path.join(os.path.expanduser("~"), ".astro_gateway_cache", name)
        os.makedirs(os.path.dirname(out), exist_ok=True)
    for src in ([DATA_URL + rel] if DATA_URL else []) + ([url] if url else []):
        try:
            import requests
            r = requests.get(src, timeout=(10, 60))
            r.raise_for_status()
            with open(out, "wb") as fh:
                fh.write(r.content)
            print(f"ดาวน์โหลด {name} จาก {src.split('/')[2]} แล้ว ({len(r.content) / 1e6:.1f} MB)")
            return out
        except Exception as e:
            print(f"  ดาวน์โหลดจาก {src.split('/')[2]} ไม่ได้ ({type(e).__name__})")
    raise FileNotFoundError(f"ไม่พบไฟล์ {name} — ตรวจการเชื่อมต่ออินเทอร์เน็ต แล้วรันเซลล์นี้ใหม่ "
                            f"(หรือดาวน์โหลดไฟล์จาก github.com/thanathi/{COURSE} โฟลเดอร์ Data/{os.path.dirname(rel) or '.'} "
                            "แล้วอัปโหลดที่แถบ Files)")


# ความยาวคลื่นขณะหยุดนิ่ง (อังสตรอม, ค่าในอากาศ)
LINES = {"Hα": 6562.8, "Hβ": 4861.3, "Hγ": 4340.5, "Hδ": 4101.7,
         "[O II] 3727": 3727.4, "[O I] 6300": 6300.3,
         "[O III] 5007": 5006.8, "[O III] 4959": 4958.9,
         "[N II] 6583": 6583.4, "[N II] 6548": 6548.0,
         "[S II] 6716": 6716.4, "[S II] 6731": 6730.8,
         "Na D": 5892.9, "Mg b": 5175.4, "Ca K": 3933.7, "Ca H": 3968.5}
C_KMS = 299792.458          # ความเร็วแสง (km/s)


def fit_line(wl, flux, center_guess, window=40, kind="auto", model=None):
    """ฟิตเกาส์เซียน + เส้นตรง กับเส้นสเปกตรัมใกล้ center_guess
    kind: "emission" (เส้นเปล่งแสง), "absorption" (เส้นดูดกลืน) หรือ "auto" (เดาจากข้อมูล)
    model: ฟังก์ชัน model(x, a, mu, s, c0, c1) ที่จะฟิต (ไม่ใส่ = เกาส์เซียน + เส้นตรงที่เตรียมไว้)
    -> (ตำแหน่งกลาง, ความคลาดเคลื่อน, ฟังก์ชันโมเดล, ค่าพารามิเตอร์, S/N ของเส้น)
    S/N ต่ำกว่า ~5 แปลว่าน่าจะฟิตได้แค่สัญญาณรบกวน ไม่ใช่เส้นจริง"""
    from scipy.optimize import curve_fit
    if model is None:
        def model(x, a, mu, s, c0, c1):
            return a * np.exp(-0.5 * ((x - mu) / s) ** 2) + c0 + c1 * (x - mu)

    wl, flux = np.asarray(wl), np.asarray(flux)
    m = (np.abs(wl - center_guess) < window) & np.isfinite(flux)
    if m.sum() < 8:                                              # หน้าต่างอยู่นอกช่วงของสเปกตรัม
        return np.nan, np.nan, model, np.full(5, np.nan), 0.0
    scale = np.nanmedian(np.abs(flux[m])) or 1.0                # ทำให้ฟลักซ์มีขนาด ~1 ก่อนฟิต (ค่าเช่น 10⁻¹³ ทำให้ฟิตไม่ขยับ)
    x, y = wl[m], flux[m] / scale
    edge = np.abs(x - center_guess) > 0.6 * window             # ขอบหน้าต่าง ใช้ประมาณระดับพื้น
    base = np.median(y[edge]) if edge.sum() >= 3 else np.median(y)
    if kind == "emission":                                       # จุดเริ่ม = ยอดสูงสุดในหน้าต่าง
        k = np.argmax(y - base)
    elif kind == "absorption":
        k = np.argmin(y - base)
    else:
        k = np.argmax(np.abs(y - base))

    p0 = [y[k] - base, x[k], min(3.0, window / 4), base, 0.0]
    bounds = ([-np.inf, x.min(), np.median(np.diff(x)), -np.inf, -np.inf], [np.inf, x.max(), window / 2, np.inf, np.inf])
    try:
        popt, pcov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=20000)
    except (RuntimeError, ValueError):                         # ฟิตไม่ลู่เข้า = ไม่พบเส้น
        return np.nan, np.nan, model, np.array(p0, float) * [scale, 1, 1, scale, scale], 0.0
    resid = y - model(x, *popt)
    noise = 1.4826 * np.median(np.abs(resid - np.median(resid)))      # ความผันผวนของพื้น (robust)
    snr = abs(popt[0]) / noise if noise > 0 else np.inf
    popt = popt * [scale, 1, 1, scale, scale]                          # คืนหน่วยเดิมของฟลักซ์
    wrong_sign = (kind == "emission" and popt[0] <= 0) or (kind == "absorption" and popt[0] >= 0)
    if wrong_sign or not (x.min() + 0.05 * window < popt[1] < x.max() - 0.05 * window):
        snr = 0.0                                                       # ผิดชนิด หรือติดขอบหน้าต่าง = ใช้ไม่ได้
    return popt[1], float(np.sqrt(pcov[1, 1])), model, popt, float(snr)


def load_spectrum(path):
    """อ่านสเปกตรัมจากไฟล์ CSV (คอลัมน์ wavelength, flux[, error]) หรือ FITS (แบบ SDSS หรือ 1 มิติพร้อม WCS)
    คืนค่า (ความยาวคลื่น [อังสตรอม], ฟลักซ์, ความคลาดเคลื่อน)"""
    path = str(path)
    if path.lower().endswith((".fits", ".fit", ".fits.gz", ".fts")):
        from astropy.io import fits
        with fits.open(path) as hdul:
            for hdu in hdul:                              # ตารางที่มี loglam (SDSS) หรือ wavelength
                cols = getattr(hdu, "columns", None)
                if cols is None:
                    continue
                low = {n.lower(): n for n in cols.names}
                d = hdu.data
                if "loglam" in low:
                    wl = 10 ** np.asarray(d[low["loglam"]], float)
                elif any(k in low for k in ("wavelength", "wave", "lambda")):
                    wl = np.asarray(d[low[next(k for k in ("wavelength", "wave", "lambda") if k in low)]], float)
                else:
                    continue
                fl = np.asarray(d[low["flux"]], float)
                if "ivar" in low:
                    iv = np.asarray(d[low["ivar"]], float)
                    err = np.where(iv > 0, 1 / np.sqrt(np.where(iv > 0, iv, 1)), np.nan)
                else:
                    err = np.asarray(d[low[next((k for k in ("error", "err", "sigma") if k in low), "flux")]], float) \
                        if any(k in low for k in ("error", "err", "sigma")) else np.full_like(fl, np.nan)
                return wl.ravel(), fl.ravel(), err.ravel()
            for hdu in hdul:                              # ภาพ 1 มิติ + WCS
                if hdu.data is not None and np.ndim(hdu.data) == 1:
                    h = hdu.header
                    step = h.get("CDELT1", h.get("CD1_1"))
                    pix = np.arange(hdu.data.size) + 1 - h.get("CRPIX1", 1)
                    wl = h["CRVAL1"] + pix * step
                    if "LOG" in str(h.get("CTYPE1", "")).upper() or h.get("DC-FLAG") == 1:
                        wl = 10 ** wl
                    fl = np.asarray(hdu.data, float)
                    return wl, fl, np.full_like(fl, np.nan)
        raise ValueError("ไม่รู้จักรูปแบบ FITS นี้ - ลองแปลงเป็น CSV (wavelength, flux)")
    df = pd.read_csv(path, sep=None, engine="python")
    low = {c.lower().strip(): c for c in df.columns}
    wcol = next(low[k] for k in ("wavelength", "wave", "lambda", "wl", "angstrom") if k in low)
    fcol = next(low[k] for k in ("flux", "f", "intensity") if k in low)
    ecol = next((low[k] for k in ("error", "err", "flux_err", "sigma") if k in low), None)
    err = df[ecol].to_numpy(float) if ecol else np.full(len(df), np.nan)
    return df[wcol].to_numpy(float), df[fcol].to_numpy(float), err


XSL_URL = "http://xsl.u-strasbg.fr/dr2_fits/"


def load_xsl(xsl_id, join=5550.0, observed=False):
    """อ่านสเปกตรัมดาวจาก X-shooter Spectral Library (XSL DR2) ซึ่งแยกเป็น 2 ไฟล์ (แขน UVB และ VIS)
    แล้วต่อเป็นเส้นเดียวที่ join Å (ปรับระดับแขน VIS ให้ต่อกับ UVB) -> (ความยาวคลื่น [Å], ฟลักซ์, error)
    XSL เลื่อนสเปกตรัมกลับไปที่ความยาวคลื่นขณะหยุดนิ่งแล้ว (ความเร็วที่ใช้เก็บใน header REST_VAL)
    observed=True เลื่อนกลับคืน ได้สเปกตรัมแบบที่วัดได้จริง (หักการโคจรของโลกรอบดวงอาทิตย์แล้ว)"""
    from astropy.io import fits
    arms = []
    for arm in ("uvb", "vis"):
        name = f"xsl_spectrum_{xsl_id}_{arm}.fits"
        with fits.open(data_path("XSL/" + name, url=XSL_URL + name)) as hdul:
            d = hdul[1].data
            wave = np.asarray(d["WAVE"], float) * 10
            if observed:
                wave = wave * (1 + hdul[0].header["REST_VAL"] / C_KMS)
            arms.append((wave, np.asarray(d["FLUX"], float), np.asarray(d["ERR"], float)))
    (wu, fu, eu), (wv, fv, ev) = arms
    near = lambda w: (w > join - 50) & (w < join)
    k = np.nanmedian(fu[near(wu)]) / np.nanmedian(fv[near(wv)])
    a, b = wu < join, wv >= join
    return np.r_[wu[a], wv[b]], np.r_[fu[a], fv[b] * k], np.r_[eu[a], ev[b] * k]

connect_drive()
print("พร้อมใช้งาน ✓")

## ตั้งค่าไฟล์ข้อมูล

In [ ]:
# ส่วน A: ดาวฤกษ์ 3 ดวงจาก X-shooter Spectral Library (XSL)
STARS = {   # ชื่อ: (รหัส XSL, ชนิดสเปกตรัม, อุณหภูมิจากแค็ตตาล็อก [K], ความเป็นโลหะ [Fe/H])
    "HD 39833":  ("X0832", "G3V", 5802, +0.13),
    "HD 160346": ("X0142", "K3V", 4926, +0.02),
    "HD 19445":  ("X0070", "G2V metal-poor", 5928, -2.04),
}

# ส่วน B: กาแล็กซีจาก SDSS (CSV: wavelength, flux หรือ FITS แบบ SDSS)
GALAXY_FILE = data_path("SDSS/spec-0531-52028-0573.fits",
    url="https://dr17.sdss.org/sas/dr17/sdss/spectro/redux/26/spectra/lite/0531/spec-0531-52028-0573.fits")
print("ไฟล์กาแล็กซี:", GALAXY_FILE)

# ส่วน A: สเปกตรัมของดาวฤกษ์
สเปกตรัมจริงของดาว 3 ดวง ถ่ายด้วยเครื่องสเปกโตรกราฟ X-shooter บนกล้อง VLT ขนาด 8.2 เมตร (ESO, ประเทศชิลี)
- **HD 39833** — ดาวแคระชนิด G3 คล้ายดวงอาทิตย์
- **HD 160346** — ดาวแคระชนิด K3 เย็นกว่าดวงอาทิตย์
- **HD 19445** — ดาวแคระที่มี **ธาตุหนัก (โลหะ) น้อยกว่าดวงอาทิตย์ประมาณ 100 เท่า**

In [ ]:
spec = {name: load_xsl(info[0]) for name, info in STARS.items()}     # อ่านทั้ง 3 ดวง

plt.figure(figsize=(12, 4.5))
for (name, (xid, sptype, teff, feh)), c in zip(STARS.items(), ["tab:orange", "tab:red", "tab:blue"]):
    w, f, e = spec[name]
    g = np.arange(3600, 9400, 5.0)
    fg = np.interp(g, w, f)
    plt.plot(g, fg / np.median(fg[(g > 5450) & (g < 5550)]), lw=0.8, color=c, label=f"{name} ({sptype})")
plt.xlabel("wavelength (Angstrom)"); plt.ylabel("flux (scaled to 1 at 5500 Å)")
plt.title("Three real stars"); plt.legend(); plt.show()

🔎 ดาวดวงไหนปล่อยแสงสีน้ำเงิน (ความยาวคลื่นสั้น) มากที่สุด? ดวงไหนเย็นที่สุด?

สเปกตรัมมี 2 ส่วน: **ความสว่างต่อเนื่อง (continuum)** ที่ค่อย ๆ เปลี่ยนตามความยาวคลื่น
และ **เส้นดูดกลืน (absorption lines)** ที่เป็นร่องแคบ ๆ — แก๊สในบรรยากาศดาวดูดกลืนแสงที่ความยาวคลื่นเฉพาะของแต่ละธาตุ
(XSL แก้ผลที่ออกซิเจนและไอน้ำใน **บรรยากาศโลก** ดูดกลืนแสงให้แล้ว แต่อาจเหลือร่องเล็ก ๆ ที่ ~6,900 และ 7,600 Å
ส่วนรอยหยักเล็ก ๆ ราว 5,550–5,750 Å คือรอยต่อของข้อมูลจาก 2 แขนของเครื่องมือ X-shooter — ทั้งสองอย่างไม่ใช่ของดาว)

## A1. เส้นสเปกตรัมนี้คือธาตุอะไร?

In [ ]:
STAR = "HD 39833"                  # 🔧 ลองเปลี่ยนเป็น "HD 160346" หรือ "HD 19445"
wl, flux, err = spec[STAR]
show = ["Ca K", "Ca H", "Hδ", "Hγ", "Hβ", "Mg b", "Na D", "Hα"]
top = np.nanpercentile(flux, 99.5)

plt.figure(figsize=(12, 4))
plt.plot(wl, flux, lw=0.4, color="k")
for i, name in enumerate(show):
    plt.axvline(LINES[name], color="tab:orange", ls=":", lw=1)
    plt.text(LINES[name], top * (1.03 + 0.07 * (i % 3)), name, ha="center", color="tab:orange")
plt.xlim(3700, 9300); plt.ylim(0, top * 1.25)
plt.xlabel("wavelength (Angstrom)"); plt.ylabel("flux (erg/s/cm²/Å)"); plt.title(STAR); plt.show()

| เส้น | ธาตุ |
|---|---|
| Hα, Hβ, Hγ, Hδ | ไฮโดรเจน (อนุกรมบัลเมอร์) |
| Na D | โซเดียม |
| Mg b | แมกนีเซียม |
| Ca H, Ca K | แคลเซียม (ไอออน) |

🔎 **คำถาม:** เส้นไหนลึกที่สุด? ลองเปลี่ยน `STAR` เป็น `"HD 160346"` แล้วรันใหม่ — เส้นไหนชัดขึ้น เส้นไหนจางลง?

## A2. ดาวที่ร้อนเท่ากัน แต่ธาตุต่างกัน
HD 39833 กับ HD 19445 มีอุณหภูมิใกล้กันมาก (5,800 และ 5,900 K) — ลองซูมดูเส้นของแคลเซียม แมกนีเซียม และเหล็ก

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, c in [("HD 39833", "tab:orange"), ("HD 19445", "tab:blue")]:
    w, f, e = spec[name]
    for a, (lo, hi) in zip(ax, [(3900, 4000), (5150, 5200)]):
        m = (w > lo) & (w < hi)
        a.plot(w[m], f[m] / np.percentile(f[m], 95), lw=0.8, color=c, label=f"{name} ([Fe/H] = {STARS[name][3]:+.2f})")
ax[0].set(title="Ca II H & K", xlabel="wavelength (Angstrom)", ylabel="flux (scaled)")
ax[1].set(title="Mg b triplet and iron lines", xlabel="wavelength (Angstrom)")
ax[1].legend(loc="lower left"); plt.show()

🔎 เส้นของโลหะใน HD 19445 **ตื้นกว่ามาก** ทั้งที่อุณหภูมิเท่ากัน → บรรยากาศของดาวมีธาตุหนักน้อย
ค่า **[Fe/H] = −2** แปลว่ามีเหล็กน้อยกว่าดวงอาทิตย์ 10² = 100 เท่า

ธาตุหนักเกิดในดาวฤกษ์รุ่นก่อน ๆ แล้วกระจายออกมาเมื่อดาวตาย → ดาวที่เกิดจากแก๊ส "ยุคแรก" ของกาแล็กซีจึงมีโลหะน้อย และมักมีอายุมาก

## A3. ชนิดสเปกตรัม สี และอุณหภูมิ
นักดาราศาสตร์จัดดาวเป็น **ชนิดสเปกตรัม O B A F G K M** (ร้อน → เย็น) จากความเข้มของเส้นสเปกตรัม
เช่น เส้นไฮโดรเจนเข้มที่สุดในดาวชนิด A ส่วนดาว K และ M มีเส้นของโลหะและโมเลกุลมาก
แต่ละชนิดแบ่งย่อย 0–9 และมีตัวอักษรบอกขนาด เช่น ดวงอาทิตย์เป็นชนิด **G2V** (V = ดาวแคระในแถบลำดับหลัก)

จากนั้นเทียบกับ **ดาวมาตรฐาน** ที่วัดอุณหภูมิอย่างละเอียดไว้แล้ว ได้ตารางว่าแต่ละชนิดมีอุณหภูมิและสีเท่าไร
ตารางที่ใช้กันมากคือของ **Pecaut & Mamajek (2013)**

In [ ]:
seq = pd.read_csv(data_path("Stars/dwarf_sequence_mamajek.csv"))     # ชนิดสเปกตรัม, อุณหภูมิ (K), สี B−V และ BP−RP, M_G, มวล
CLASSES = "OBAFGKM"
seq["x"] = [10 * CLASSES.index(t[0]) + float(t[1:].rstrip("V")) for t in seq.SpT]   # G2V → 42 ใช้วาดแกน x
seq[seq.SpT.isin(["B5V", "A0V", "F0V", "G2V", "K0V", "M0V", "M5V"])]

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].scatter(seq.x, seq.Teff, c=seq.Teff, cmap="RdYlBu", vmin=2600, vmax=12000, edgecolor="k", lw=0.3)
ax[1].scatter(seq.x, seq.BP_RP, c=seq.Teff, cmap="RdYlBu", vmin=2600, vmax=12000, edgecolor="k", lw=0.3)
for name, (xid, sptype, teff, feh) in STARS.items():          # ดาว 3 ดวงของเรา ตามชนิดสเปกตรัมในแค็ตตาล็อก
    x = 10 * CLASSES.index(sptype[0]) + float(sptype[1])
    ax[0].plot(x, teff, "o", ms=10, mfc="none", mew=2, label=f"{name} ({sptype})")
ax[0].set_yscale("log"); ax[0].set_ylabel("temperature (K)"); ax[0].legend()
ax[1].set_ylabel("colour BP - RP (mag)")
ax[1].set_xticks([10 * i + 5 for i in range(7)]); ax[1].set_xticklabels(list(CLASSES))
ax[1].set_xlabel("spectral type (hot -> cool)")
plt.show()

🔎 ดาวชนิดไหนร้อนที่สุด? ชนิดไหนสีแดงที่สุด? ดวงอาทิตย์ (G2) อยู่ตรงไหน?

### หาอุณหภูมิจากชนิดสเปกตรัม
อ่านอุณหภูมิของชนิดสเปกตรัมนั้นจากตาราง

In [ ]:
table = seq.set_index("SpT")
STAR = "HD 39833"                                  # 🔧 ลองเปลี่ยนเป็น "HD 160346" หรือ "HD 19445"
sptype = STARS[STAR][1].split()[0]                 # เช่น "G3V"
T_spt = ____                       # ✏️ อุณหภูมิของชนิดนี้จากตาราง (ใช้ table.Teff[...])
print(f"{STAR} ชนิด {sptype} → อุณหภูมิ ≈ {T_spt:,} K  (ค่าจากแค็ตตาล็อก {STARS[STAR][2]:,} K)")

### หาอุณหภูมิจากสีที่ Gaia วัดได้
ยาน Gaia วัดสี BP − RP ของดาวเกือบ 2 พันล้านดวง — ถ้ารู้สี ก็อ่านอุณหภูมิจากเส้นในตารางได้ทันที
ใช้ `np.interp(ค่าที่รู้, แกน x ของตาราง, แกน y ของตาราง)` อ่านค่าระหว่างจุดในตาราง (interpolation)

In [ ]:
BP_RP = {"HD 39833": 0.791, "HD 160346": 1.182, "HD 19445": 0.690}     # จาก Gaia DR3
ref = seq.dropna(subset=["BP_RP"]).query("BP_RP < 4").sort_values("BP_RP")   # np.interp ต้องเรียงแกน x จากน้อยไปมาก

for name in STARS:
    T_colour = ____        # ✏️ อ่านอุณหภูมิจากสี
    print(f"{name:10s} BP-RP = {BP_RP[name]:.3f} → อุณหภูมิ ≈ {T_colour:,.0f} K  (ค่าจากแค็ตตาล็อก {STARS[name][2]:,} K)")

plt.figure(figsize=(7, 4.5))
plt.plot(ref.BP_RP, ref.Teff, "-", color="gray", label="standard stars")
for name in STARS:
    plt.plot(BP_RP[name], STARS[name][2], "o", ms=9, label=f"{name} (catalogue)")
plt.xlim(0.3, 1.6); plt.ylim(4000, 7000)
plt.xlabel("colour BP - RP (mag)"); plt.ylabel("temperature (K)"); plt.legend(); plt.show()

🔎 **คิดต่อ**
1. วิธีไหน (ชนิดสเปกตรัม หรือสี) ได้ใกล้ค่าในแค็ตตาล็อกกว่ากัน? ค่าในแค็ตตาล็อกได้จากการฟิตสเปกตรัมทั้งเส้นกับแบบจำลองบรรยากาศดาว ซึ่งละเอียดที่สุด
2. HD 19445 ได้อุณหภูมิจากสี **สูงกว่าความจริง** — ใบ้: ดูขั้น A2 เส้นของโลหะดูดกลืนแสงสีอะไรมาก? ถ้าโลหะน้อย ดาวจะดูสีอะไร?
3. ดาวดวงไหนร้อนกว่าหรือเย็นกว่าดวงอาทิตย์ (~5,800 K)? เข้ากับสีของสเปกตรัมในภาพแรกไหม?

## A5. ดาวเคลื่อนที่เข้าหาหรือออกห่างเรา? (ความเร็วแนวเล็ง)
ดาวทุกดวงในกาแล็กซีกำลังเคลื่อนที่ ถ้าดาว **เคลื่อนที่ออกห่าง** แสงจะถูกยืดออก เส้นสเปกตรัมเลื่อนไปทาง **แดง** (ความยาวคลื่นยาวขึ้น)
ถ้า **เคลื่อนที่เข้าหา** เส้นจะเลื่อนไปทาง **น้ำเงิน** — เรียกว่า **ปรากฏการณ์ดอปเพลอร์** (Doppler effect)

**ความเร็วแนวเล็ง (radial velocity): v = c × (λ_สังเกต − λ_หยุดนิ่ง) / λ_หยุดนิ่ง**  (v บวก = ออกห่าง, v ลบ = เข้าหา)

ดาวเคลื่อนที่ช้ากว่าแสงมาก เส้นจึงเลื่อนไปแค่ไม่กี่อังสตรอม ต้องซูมดูใกล้ ๆ

> 📝 คลังข้อมูล XSL เลื่อนสเปกตรัมทุกดวงกลับไปที่ตำแหน่ง "หยุดนิ่ง" ไว้แล้ว เพื่อให้เทียบดาวต่างดวงกันง่าย (ขั้น A1–A4 ใช้แบบนี้)
> และจดความเร็วที่ใช้เลื่อนไว้ในไฟล์ — `load_xsl(..., observed=True)` จะเลื่อนกลับคืน ได้สเปกตรัมแบบที่กล้องวัดได้จริง
> (หักผลจากการที่โลกโคจรรอบดวงอาทิตย์ด้วยความเร็ว ~30 km/s ออกแล้ว)

In [ ]:
obs = {name: load_xsl(info[0], observed=True) for name, info in STARS.items()}    # สเปกตรัมแบบที่วัดได้จริง

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for i, (name, c) in enumerate(zip(STARS, ["tab:orange", "tab:red", "tab:blue"])):
    w, f, e = obs[name]
    for a, (lo, hi) in zip(ax, [(5880, 5902), (6550, 6575)]):
        m = (w > lo) & (w < hi)
        a.plot(w[m], f[m] / np.percentile(f[m], 95) - i, lw=1, color=c, label=name)    # เลื่อนกราฟลงทีละ 1 ไม่ให้ทับกัน
for a, rest in zip(ax, [[5889.95, 5895.92], [6562.80]]):
    for r in rest:
        a.axvline(r, color="k", ls=":", lw=1)
ax[0].set(title="Na D (sodium)", xlabel="wavelength (Angstrom)", ylabel="flux (scaled, offset)")
ax[1].set(title="H-alpha (hydrogen)", xlabel="wavelength (Angstrom)")
ax[1].legend(loc="lower right"); plt.show()

🔎 เส้นจุดสีดำคือตำแหน่งเมื่อหยุดนิ่ง — ดาวดวงไหนเส้นเลื่อนไปมากที่สุด? เลื่อนไปทางไหน แปลว่าดาวเคลื่อนที่เข้าหาหรือออกห่างเรา?

### วัดความเร็วให้แม่นด้วยหลายเส้น
ทำเหมือนการวัด redshift ของกาแล็กซีในส่วน B: เดาความเร็วด้วยตาก่อน (`V_GUESS`) แล้วฟิตเกาส์เซียนกับเส้นดูดกลืนหลายเส้น
เส้นของดาวแคบมาก จึงใช้หน้าต่างฟิตแคบ (±1.5 Å) กันไม่ให้ไปโดนเส้นข้างเคียง — ถ้าเดาผิดไปมาก จะหาเส้นไม่เจอ

In [ ]:
STAR = "HD 39833"                  # 🔧 ลองทั้ง 3 ดวง (HD 19445 ต้องปรับ V_GUESS ด้วย)
V_GUESS = 0                        # 🔧 ความเร็วที่เดา (km/s) ปรับจนเส้นประสีส้มตรงกับร่องของเส้น Na D
wl, flux, err = obs[STAR]

m = (wl > 5880) & (wl < 5902)
plt.figure(figsize=(10, 3.5))
plt.plot(wl[m], flux[m], color="k", lw=1)
for r in [5889.95, 5895.92]:
    plt.axvline(r * (1 + V_GUESS / C_KMS), color="tab:orange", ls="--")
plt.xlabel("wavelength (Angstrom)"); plt.ylabel("flux"); plt.title(f"{STAR}: v guess = {V_GUESS} km/s"); plt.show()

In [ ]:
# เส้นดูดกลืนที่ใช้วัด: ชื่อ -> (ความยาวคลื่นขณะหยุดนิ่งในอากาศ [Å], ครึ่งความกว้างหน้าต่างฟิต [Å])
RV_LINES = {"Mg b 5167": (5167.32, 1.5), "Mg b 5173": (5172.68, 1.5), "Mg b 5184": (5183.60, 1.5),
            "Na D2 5890": (5889.95, 1.5), "Na D1 5896": (5895.92, 1.5), "Hα 6563": (6562.80, 8),
            "Ca II 8498": (8498.02, 3), "Ca II 8542": (8542.09, 3), "Ca II 8662": (8662.14, 3)}
v_center = V_GUESS
for round_ in range(2):                                  # รอบที่ 2 เลื่อนหน้าต่างฟิตไปไว้กลางเส้นตามผลรอบแรก
    rows = []
    for name, (rest, win) in RV_LINES.items():
        pred = rest * (1 + v_center / C_KMS)             # ตำแหน่งที่คาด
        c, ce, _, _, s = fit_line(wl, flux, pred, window=win, kind="absorption")
        if s >= 5:
            v_line = ____         # ✏️ v = c × (λ_สังเกต − λ_หยุดนิ่ง) / λ_หยุดนิ่ง  (c, rest เป็น Å)
            rows.append({"line": name, "rest (Å)": rest, "observed (Å)": c, "v (km/s)": v_line})
    rv = pd.DataFrame(rows)
    rv_ok = len(rv) >= 3 and rv["v (km/s)"].std() < 10    # ทุกเส้นให้ค่าใกล้กัน = ฟิตถูกเส้น
    if not rv_ok:
        break
    v_center = rv["v (km/s)"].mean()

if not rv_ok:
    print("⚠️ หาเส้นไม่เจอ หรือแต่ละเส้นให้ค่าต่างกันมาก (ฟิตโดนเส้นอื่น) — ปรับ V_GUESS ให้เส้นประตรงกับร่องของเส้น Na D แล้วรันใหม่")
else:
    print(rv.round(2).to_string(index=False))
    v_star, v_err = rv["v (km/s)"].mean(), rv["v (km/s)"].std() / np.sqrt(len(rv))
    print(f"\n{STAR}: ความเร็วแนวเล็ง = {v_star:+.1f} ± {v_err:.1f} km/s  ->  ดาวกำลังเคลื่อนที่" + ("ออกห่างจากเรา" if v_star > 0 else "เข้าหาเรา"))

เทียบกับค่าที่นักดาราศาสตร์วัดด้วยเครื่องมือความละเอียดสูง:

| ดาว | ความเร็วแนวเล็งอ้างอิง (km/s) | ที่มา |
|---|---|---|
| HD 39833 | +24.9 | ดาวมาตรฐานความเร็วแนวเล็งของ Gaia (Soubiran et al. 2018) |
| HD 160346 | +15 ถึง +26 (เปลี่ยนไปมาทุก 83.7 วัน) | วงโคจรดาวคู่ในแค็ตตาล็อก SB9 |
| HD 19445 | −140.0 | Soubiran et al. 2018 |

🔎 **คิดต่อ**
1. ค่าที่วัดได้ใกล้ค่าอ้างอิงแค่ไหน? HD 39833 ต่างไปราว 7 km/s ทั้งที่ความคลาดเคลื่อนจากการฟิตไม่ถึง 1 km/s
   ความคลาดเคลื่อนที่ทำให้ผลเพี้ยนไปทางเดียวกันทุกเส้นแบบนี้เรียกว่า **ความคลาดเคลื่อนเชิงระบบ (systematic error)** — ลองเดาว่ามาจากอะไรได้บ้าง
2. **HD 160346 เป็นดาวคู่** มีดาวข้างเคียงที่มองไม่เห็นโคจรรอบ ดึงให้ดาวส่ายไปมา ความเร็วแนวเล็งจึงเปลี่ยนเป็นคาบ 83.7 วัน
   หลักการเดียวกับการชั่งน้ำหนักดาวเคราะห์ในโน้ตบุ๊ก 03 (ขั้น B4) แต่ดาวเคราะห์ทำให้ดาวแม่ส่ายแค่ระดับเมตรต่อวินาที
3. **HD 19445 พุ่งเข้าหาเราเร็ว 140 km/s** ทั้งที่อยู่ห่างเราพอ ๆ กับ HD 39833 (~39 พาร์เซก)
   ดวงอาทิตย์และดาวส่วนใหญ่แถวนี้อยู่ใน **จาน (disk)** ของทางช้างเผือก โคจรรอบใจกลางกาแล็กซีไปพร้อม ๆ กัน ความเร็วเทียบกันจึงแค่หลักสิบ km/s
   ส่วน HD 19445 เป็นดาวใน **ฮาโล (halo)** ที่โคจรคนละแบบ จึงพุ่งผ่านจานด้วยความเร็วสูง — และเป็นดาวโลหะน้อย (ขั้น A2) ซึ่งเป็นดาวรุ่นแรก ๆ ของกาแล็กซี
4. ดาวเทียม Gaia วัดความเร็วแนวเล็งของดาวกว่า 33 ล้านดวง โดยใช้เส้นแคลเซียมช่วง 8,450–8,720 Å — ชุดเดียวกับ Ca II 8498, 8542, 8662 ที่เราใช้

In [ ]:
# (เลือกทำ) ความเร็วในอวกาศ = ความเร็วแนวเล็ง (จากสเปกตรัม) รวมกับความเร็วตัดขวาง (จาก proper motion ของ Gaia เหมือนโน้ตบุ๊ก 01)
# ความเร็วตัดขวาง v_t = 4.74 × μ × d   (μ เป็นพิลิปดาต่อปี, d เป็นพาร์เซก, v_t เป็น km/s)
GAIA = {"HD 39833": (25.06, 116.0, 24.7),     # พารัลแลกซ์ (mas), proper motion รวม (mas/ปี), ความเร็วแนวเล็ง (km/s) จาก Gaia DR3
        "HD 19445": (25.87, 855.7, -140.2)}   # 🔧 แทนความเร็วแนวเล็งด้วยค่าที่วัดเองได้
for name, (plx, pm, v_r) in GAIA.items():
    d = ____                            # ✏️ ระยะทาง (พาร์เซก)
    v_t = ____              # ✏️ ความเร็วตัดขวาง (km/s) — ระวังหน่วยของ μ
    print(f"{name}: ระยะ {d:.0f} pc · v_r = {v_r:+.0f} km/s · v_t = {v_t:.0f} km/s · ความเร็วรวม ≈ {np.hypot(v_r, v_t):.0f} km/s")

# ส่วน B: กาแล็กซีเคลื่อนที่ออกห่างเราเร็วแค่ไหน?

In [ ]:
gwl, gflux, gerr = load_spectrum(GALAXY_FILE)

plt.figure(figsize=(12, 4))
plt.plot(gwl, gflux, lw=0.7, color="k")
plt.xlabel("wavelength (Angstrom)"); plt.ylabel("flux")
plt.title("Galaxy spectrum"); plt.show()

นี่คือสเปกตรัมจริงของกาแล็กซี **SDSS J140102.07+025242.5** ในกลุ่มดาวหญิงสาว (Virgo)
ถ่ายด้วยกล้องโทรทรรศน์ 2.5 เมตรของโครงการ SDSS ที่หอดูดาว Apache Point สหรัฐอเมริกา
(ยอดแหลมเล็ก ๆ ทางขวาสุดส่วนใหญ่เป็นแสงรบกวนจากชั้นบรรยากาศโลก)

กาแล็กซีนี้มี **เส้นเปล่งแสง (emission lines)** จากแก๊สร้อนที่ถูกดาวเกิดใหม่กระตุ้น
แต่เส้นเหล่านี้ **ไม่ได้อยู่ที่ความยาวคลื่นเดิม** — ถูกเลื่อนไปทางแดง เพราะกาแล็กซีกำลังเคลื่อนที่ออกห่างจากเรา

**redshift: z = (λ_สังเกต − λ_หยุดนิ่ง) / λ_หยุดนิ่ง**  ·  **ความเร็ว v ≈ c × z** (เมื่อ z น้อย ๆ)

## B1. ประมาณ z ด้วยตา
เปลี่ยนค่า `Z_GUESS` แล้วรันใหม่ จนเส้นประสีส้มตรงกับยอดเส้นเปล่งแสง (ลองเริ่มจาก `0` ก่อน จะเห็นว่าไม่ตรงเลย)

In [ ]:
Z_GUESS = 0                         # ✏️ ลอง 0.1, 0.2, 0.3, ... จนเส้นประตรงกับยอดเส้นเปล่งแสง

names = ["[O II] 3727", "Hβ", "[O III] 4959", "[O III] 5007", "[O I] 6300", "[N II] 6548", "Hα", "[N II] 6583", "[S II] 6716", "[S II] 6731"]
plt.figure(figsize=(12, 4))
plt.plot(gwl, gflux, lw=0.7, color="k")
for name in names:
    plt.axvline(LINES[name] * (1 + Z_GUESS), color="tab:orange", ls="--", lw=1)
plt.xlabel("wavelength (Angstrom)"); plt.ylabel("flux"); plt.title(f"z guess = {Z_GUESS}"); plt.show()

## B2. วัดตำแหน่งเส้นให้แม่นด้วยการฟิตเกาส์เซียน
เส้นสเปกตรัมมีรูปร่างคล้าย **ระฆังคว่ำ (ฟังก์ชันเกาส์เซียน)** วางอยู่บนพื้นที่ค่อย ๆ เปลี่ยน

**เกาส์เซียน: a × exp( −(x − μ)² / (2s²) )**  — a = ความสูงของเส้น, μ = ตำแหน่งกลาง, s = ความกว้าง

คอมพิวเตอร์จะปรับค่า a, μ, s ไปเรื่อย ๆ จนเส้นโค้งทับข้อมูลได้ดีที่สุด แล้วเราอ่านค่า μ เป็นตำแหน่งของเส้น

In [ ]:
def gaussian(x, a, mu, s, c0, c1):
    """เส้นสเปกตรัม = เกาส์เซียน (ความสูง a, ตำแหน่งกลาง mu, ความกว้าง s) + พื้นเป็นเส้นตรง c0 + c1 (x − mu)"""
    peak = ____      # ✏️ เขียนสมการเกาส์เซียน (ใช้ np.exp และ ** สำหรับยกกำลัง)
    return peak + c0 + c1 * (x - mu)

xx = np.linspace(-5, 5, 200)                      # ตรวจสอบ: ควรได้รูประฆัง ยอดสูง 1 ที่ x = 0 และเกือบเป็นศูนย์เมื่อห่างจาก 0 เกิน 3
plt.figure(figsize=(5, 2.5)); plt.plot(xx, gaussian(xx, 1, 0, 1, 0, 0)); plt.title("check: Gaussian a=1, mu=0, s=1"); plt.show()

In [ ]:
LINE = "Hα"                                     # เส้นที่จะวัด
expected = LINES[LINE] * (1 + Z_GUESS)
center, center_err, model, popt, snr = fit_line(gwl, gflux, expected, window=25, kind="emission", model=gaussian)   # หน้าต่าง ±25 Å รอบตำแหน่งที่คาด
z = ____                   # ✏️ z = (λ_สังเกต − λ_หยุดนิ่ง) / λ_หยุดนิ่ง
z_err = center_err / LINES[LINE]

m = np.abs(gwl - center) < 40
xx = np.linspace(center - 25, center + 25, 400)
plt.figure(figsize=(7, 4))
plt.plot(gwl[m], gflux[m], "o", ms=3, color="k", label="data")
plt.plot(xx, model(xx, *popt), color="tab:orange", label="Gaussian fit")
plt.axvline(center, color="tab:orange", ls=":")
plt.xlabel("wavelength (Angstrom)"); plt.ylabel("flux"); plt.legend(); plt.show()

if snr < 5:
    print(f"⚠️ ไม่พบเส้นที่ชัดเจน (S/N = {snr:.1f}) — ปรับ Z_GUESS ในขั้น B1 ให้เส้นประตรงกับยอดก่อน")
print(f"{LINE}: λ_สังเกต = {center:.2f} ± {center_err:.2f} Å (S/N = {snr:.0f})  ->  z = {z:.5f} ± {z_err:.5f}")

## B3. ความเร็วและระยะทาง (กฎของฮับเบิล)
กาแล็กซีที่อยู่ไกลกว่าเคลื่อนที่ออกห่างเร็วกว่า: **v = H₀ × d** โดย H₀ ≈ 70 km/s ต่อเมกะพาร์เซก (Mpc)

In [ ]:
if snr < 5:
    print("⚠️ ขั้น B2 ยังไม่ผ่าน (S/N < 5) — ค่าที่ได้ด้านล่างยังใช้ไม่ได้ ปรับ Z_GUESS ในขั้น B1 ก่อน")
H0 = 70.0                                  # km/s/Mpc
v = ____                              # ✏️ ความเร็วถอยห่าง (km/s)
d_mpc = ____                             # ✏️ ระยะทาง (เมกะพาร์เซก)
print(f"ความเร็วถอยห่าง ≈ {v:,.0f} km/s")
print(f"ระยะทาง ≈ {d_mpc:,.0f} เมกะพาร์เซก ≈ {d_mpc * 3.26:,.0f} ล้านปีแสง")

> ⚠️ กาแล็กซีนี้มี z ≈ 0.25 ซึ่ง **ไม่น้อยแล้ว** — สูตร v ≈ c × z และ v = H₀ × d เป็นค่าประมาณคร่าว ๆ
> นักดาราศาสตร์ใช้แบบจำลองการขยายตัวของเอกภพ (จักรวาลวิทยา) คำนวณให้แม่นขึ้น ลองดูเซลล์ถัดไป

In [ ]:
# (เลือกทำ) ใช้แบบจำลองจักรวาลวิทยาจาก astropy: แสงจากกาแล็กซีนี้ออกเดินทางมานานเท่าไร?
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)          # เอกภพแบนที่มีสสาร 30% พลังงานมืด 70%
print(f"แสงออกเดินทางมาแล้ว ≈ {cosmo.lookback_time(z).value:.1f} พันล้านปี (โลกอายุ 4.6 พันล้านปี)")
print(f"ระยะทางปัจจุบัน (comoving) ≈ {cosmo.comoving_distance(z).value:,.0f} Mpc")

## B4. ใช้หลายเส้นช่วยยืนยัน
ใช้ z จากขั้น B2 ทำนายตำแหน่งเส้นอื่น แล้วฟิตในหน้าต่างแคบ ±15 Å (กันไม่ให้ไปฟิตโดนเส้นข้างเคียง)

In [ ]:
rows = []
if snr < 5:
    print("⚠️ ทำขั้น B2 ให้ผ่านก่อน (ต้องได้ S/N ≥ 5) แล้วค่อยรันเซลล์นี้")
else:
    for name in ["[O II] 3727", "Hβ", "[O III] 5007", "[O I] 6300", "Hα", "[N II] 6583", "[S II] 6716"]:
        pred = LINES[name] * (1 + z)                         # ตำแหน่งที่ทำนายจาก z ของขั้น B2
        c, ce, _, _, s = fit_line(gwl, gflux, pred, window=15, kind="emission")
        if s >= 5 and abs(c - pred) < 5:
            rows.append({"line": name, "observed": c, "z": c / LINES[name] - 1, "S/N": s})
        else:
            print(f"{name}: ไม่พบเส้นที่ชัดเจน ข้าม")
if len(rows) >= 3:
    tab = pd.DataFrame(rows)
    print(tab.round(5).to_string(index=False))
    print(f"\nz เฉลี่ย = {tab.z.mean():.5f} ± {tab.z.std() / np.sqrt(len(tab)):.5f}")
elif snr >= 5:
    print(f"⚠️ ยืนยันได้แค่ {len(rows)} เส้น — เส้นที่ฟิตในขั้น B2 น่าจะไม่ใช่ Hα (อาจเป็น [N II] หรือ [O I])")
    print("   กลับไปปรับ Z_GUESS ในขั้น B1 ให้เส้นประตรงกับยอดทุกเส้น แล้วรัน B2 ใหม่")

🔎 **คำถามชวนคิด**
1. ทำไมแต่ละเส้นให้ค่า z ต่างกันเล็กน้อย?
2. ถ้าเส้นเลื่อนไปทาง **น้ำเงิน** (z ติดลบ) แปลว่าอะไร? (ใบ้: กาแล็กซีแอนดรอเมดา)
3. ตอนที่แสงจากกาแล็กซีนี้ออกเดินทาง บนโลกมีอะไรอยู่บ้าง?
4. ค่า z ที่ SDSS วัดได้คือ 0.25196 — ของเราต่างไปเท่าไร? (ดูหมายเหตุด้านล่าง)

> 📝 SDSS ใช้ความยาวคลื่นที่วัดในสุญญากาศ ส่วนตาราง `LINES` เป็นค่าในอากาศ
> ต่างกันประมาณ 0.03% (z คลาดไป ~0.0003) — พอใช้ได้สำหรับกิจกรรมนี้